In [ ]:
import os
from pathlib import Path
import pandas as pd
import gradio as gr
from openai import OpenAI



class Config:
    BASE_URL = "https://api.groq.com/openai/v1"
    MODEL = "openai/gpt-oss-120b"
    DATA_PATH = Path.cwd().parent / "data" / "raw" / "german_credit_data.csv"
    API_KEY = os.getenv("API_KEY")




def initialize_client():
    if not Config.API_KEY:
        raise ValueError("API_KEY environment variable not set.")
    
    return OpenAI(
        base_url=Config.BASE_URL,
        api_key=Config.API_KEY
    )




def load_dataset():
    try:
        df = pd.read_csv(Config.DATA_PATH)
        return df
    except Exception as e:
        raise RuntimeError(f"Failed to load dataset: {e}")




def create_prompt(row):
    return f"""
You are explaining credit risk to a normal person with no finance knowledge.

Customer details:
Age: {row['Age']}
Job: {row['Job']}
Housing: {row['Housing']}
Savings: {row['Saving accounts']}
Checking balance: {row['Checking account']}
Loan amount: {row['Credit amount']}
Duration: {row['Duration']}
Purpose: {row['Purpose']}

Risk: {row['Risk']}

Explain in very simple and clear language why this person is considered {row['Risk']}.
Use easy words like talking to a beginner. Keep it short (3–5 lines).
If the risk is bad, give 1–2 simple suggestions to improve.
"""



class CreditRiskAnalyzer:
    def __init__(self, df, client):
        self.df = df
        self.client = client

    def analyze(self, index):
        try:
            index = int(index)

            if index < 0 or index >= len(self.df):
                return "Invalid index. Please enter a valid row number."

            row = self.df.iloc[index]
            prompt = create_prompt(row)

            response = self.client.chat.completions.create(
                model=Config.MODEL,
                messages=[
                    {"role": "system", "content": "You are an expert financial analyst."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.7
            )

            return response.choices[0].message.content

        except Exception as e:
            return f"Error: {str(e)}"



def create_interface(analyzer):
    return gr.Interface(
        fn=analyzer.analyze,
        inputs=gr.Textbox(label="Enter Row Index"),
        outputs=gr.Textbox(label="Explanation"),
        title="💳 Credit Risk Explainer",
        description="Enter a dataset row index to understand the customer's credit risk in simple language."
    )




def main():
    try:
        print("Initializing application...")

        # Initialize components
        client = initialize_client()
        df = load_dataset()
        analyzer = CreditRiskAnalyzer(df, client)

        # Launch UI
        app = create_interface(analyzer)
        app.launch()

    except Exception as e:
        print(f"Application failed to start: {e}")


# ==============================
# Run App
# ==============================
if __name__ == "__main__":
    main()

: 